In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)

# โหลด Dataset
df = pd.read_csv('test_set (True Unseen).csv')
print(df.head(5))

path = "D:/New Finetune Hackathon/Finetuned Bert Model State 2/checkpoint-78"
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
def add_prefix_token(text):
    # clean log
    text = str(text).replace("\t", " ").strip()

    # add prefix token
    if text[0].isalpha() or text[3].isalpha():
        return "[SQL]\n" + text
    else:
        return "[LOG]\n" + text

In [ ]:
# Label mapping (ต้องตรงกับตอน train)
label_map = {
    "ANOMALY": 0,
    "NORMAL": 1
}

In [ ]:
def predict_log_with_loss(log_text, true_label):
    log_text = add_prefix_token(log_text)

    inputs = tokenizer(
        log_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    label_id = torch.tensor([label_map[true_label]]).to(device)

    with torch.no_grad():
        outputs = model(**inputs, labels=label_id)
        loss = outputs.loss
        logits = outputs.logits

        pred = torch.argmax(logits, dim=1).item()
        prob = torch.softmax(logits, dim=-1).tolist()[0]

    prediction = "NORMAL" if pred == 1 else "ANOMALY"
    return prediction, prob, loss.item()

In [ ]:
correct_predictions = 0
total_predictions = len(df)
total_val_loss = 0.0

y_true = []
y_pred = []

for index, row in df.iterrows():
    text_to_classify = row['query log']
    true_label = row['status']

    prediction_result, confidence, loss = predict_log_with_loss(
        text_to_classify, true_label
    )

    total_val_loss += loss
    y_true.append(true_label)
    y_pred.append(prediction_result)

    if prediction_result == true_label:
        correct_predictions += 1
        correction = 'True'
    else:
        correction = 'False'

    print(
        f"prediction = {prediction_result} | "
        f"true_status = {true_label} | "
        f"correction = {correction} | "
        f"confidence = {confidence}"
    )

# Accuracy
accuracy = (correct_predictions / total_predictions) * 100

# Precision / Recall / F1 (ANOMALY = positive class)
precision = precision_score(y_true, y_pred, pos_label="ANOMALY")
recall = recall_score(y_true, y_pred, pos_label="ANOMALY")
f1 = f1_score(y_true, y_pred, pos_label="ANOMALY")

# Validation Loss
avg_val_loss = total_val_loss / total_predictions

# ===============================
# Results
# ===============================
print("\n=== Evaluation Results ===")
print(f"Total samples   : {total_predictions}")
print(f"Accuracy        : {accuracy:.2f}%")
print(f"Precision       : {precision:.4f}")
print(f"Recall          : {recall:.4f}")
print(f"F1-score        : {f1:.4f}")
print(f"Validation Loss : {avg_val_loss:.4f}")